# Tabular Model Baselines

This notebook builds tabular baselines for comparison with the Book-of-Life text approach.

We use the same clean temporal setup as `BoL approach 2`:

- **Features:** stable background variables plus pre-2000 predictors
- **Target:** `later_persistent_delinquency_contact_2000_2020`
- **Outcome window:** 2000-2020

The goal is to compare two representations of the same information:

- BoL: predictors rendered as life-history text
- Tabular models: predictors used as a numeric feature matrix

## Models

This notebook implements two tabular models.

**1. L2 Logistic Regression**

An interpretable linear baseline. It is useful because it shows how much signal is available from a regularized linear model.

**2. Gradient Boosting with Decision Stumps**

A lightweight nonlinear baseline. It can pick up simple threshold effects and repeated interactions better than plain logistic regression.

Both implementations use only NumPy and Pandas so the notebook can run without scikit-learn.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

cwd = Path.cwd().resolve()
if cwd.name == "tabular_models" and cwd.parent.name == "BoL approach 2":
    TABULAR_DIR = cwd
elif cwd.name == "BoL approach 2":
    TABULAR_DIR = cwd / "tabular_models"
else:
    TABULAR_DIR = cwd / "BoL approach 2" / "tabular_models"

APPROACH_DIR = TABULAR_DIR.parent
PROJECT_DIR = APPROACH_DIR.parent

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
FEATURE_INDEX_PATH = APPROACH_DIR / "data" / "features" / "baseline_pre2000_bol_feature_index.csv"
TARGETS_PATH = APPROACH_DIR / "data" / "targets" / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
OUT_DIR = TABULAR_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "later_persistent_delinquency_contact_2000_2020"
MISSING_CODES = {-1, -2, -3, -4, -5, -7}
RANDOM_SEED = 2026
TEST_SIZE = 0.30

## Load Data

The raw feature values come from `nlsy79_child_youngadult_selected_crime_features.csv`.

The feature list comes from `baseline_pre2000_bol_feature_index.csv`, which contains only XRND/static variables and features before 2000.

The target file contains the temporally defined delinquency/contact targets from 2000-2020.

In [ ]:
data = pd.read_csv(DATA_PATH)
feature_index = pd.read_csv(FEATURE_INDEX_PATH)
targets = pd.read_csv(TARGETS_PATH)

feature_cols = [c for c in feature_index["csv_code"].tolist() if c in data.columns]

model_df = data[["C0000100"] + feature_cols].merge(
    targets[["C0000100", TARGET]],
    on="C0000100",
    how="inner",
)
model_df = model_df[model_df[TARGET].notna()].copy()

print("Rows:", len(model_df))
print("Features:", len(feature_cols))
print("Target balance:")
print(model_df[TARGET].value_counts(normalize=False).sort_index())
print("Base rate:", round(model_df[TARGET].mean(), 3))

## Preprocessing

The tabular models need a numeric matrix. We therefore:

1. convert feature columns to numeric values,
2. replace NLSY missing codes (`-1, -2, -3, -4, -5, -7`) with missing values,
3. impute missing values with the training median,
4. standardize features for logistic regression.

This differs from BoL, where missing values are skipped and observed values are rendered as text.

In [ ]:
def clean_feature_matrix(df, feature_cols):
    x = df[feature_cols].copy()
    for col in feature_cols:
        x[col] = pd.to_numeric(x[col], errors="coerce")
        x[col] = x[col].replace([np.inf, -np.inf], np.nan)
        x.loc[x[col].isin(MISSING_CODES), col] = np.nan
    return x


def train_test_split_stratified(y, test_size=0.30, seed=2026):
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    for label in sorted(np.unique(y)):
        idx = np.where(y == label)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test].tolist())
        train_idx.extend(idx[n_test:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return np.array(train_idx), np.array(test_idx)


def fit_median_imputer(x_train):
    return x_train.median(axis=0, skipna=True).fillna(0.0)


def impute_with_median(x, medians):
    arr = x.fillna(medians).to_numpy(dtype=float)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def fit_standardizer(x):
    means = x.mean(axis=0)
    stds = x.std(axis=0)
    stds[stds == 0] = 1.0
    return means, stds


def standardize(x, means, stds):
    z = (x - means) / stds
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0), -10.0, 10.0)

In [ ]:
y = model_df[TARGET].astype(int).to_numpy()
x_raw = clean_feature_matrix(model_df, feature_cols)

train_idx, test_idx = train_test_split_stratified(y, TEST_SIZE, RANDOM_SEED)

x_train_raw = x_raw.iloc[train_idx]
x_test_raw = x_raw.iloc[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

medians = fit_median_imputer(x_train_raw)
x_train_imputed = impute_with_median(x_train_raw, medians)
x_test_imputed = impute_with_median(x_test_raw, medians)

means, stds = fit_standardizer(x_train_imputed)
x_train_std = standardize(x_train_imputed, means, stds)
x_test_std = standardize(x_test_imputed, means, stds)

print("Train N:", len(y_train), "Test N:", len(y_test))
print("Train base rate:", round(y_train.mean(), 3))
print("Test base rate:", round(y_test.mean(), 3))

## Shared Metrics

In [ ]:
def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def auc_score(y_true, prob):
    order = np.argsort(prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(prob) + 1)
    pos = y_true == 1
    n_pos = int(pos.sum())
    n_neg = int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    rank_sum_pos = ranks[pos].sum()
    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    return {
        "threshold": threshold,
        "accuracy": float((pred == y_true).mean()),
        "auc": auc_score(y_true, prob),
        "sensitivity_tpr": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity_tnr": tn / (tn + fp) if (tn + fp) else np.nan,
        "predicted_positive_rate": float(pred.mean()),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

# Model 1: L2 Logistic Regression

The logistic regression model is the main interpretable tabular baseline. It uses standardized features and an L2 penalty.

The L2 penalty helps because many pre-2000 features are repeated versions of similar questions across survey years.

In [ ]:
L2_STRENGTH = 1.0
LOGIT_LEARNING_RATE = 0.01
LOGIT_MAX_ITER = 5000
LOGIT_TOL = 1e-8


def linear_predict(x_design, beta):
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        z = x_design @ beta
    return np.nan_to_num(z, nan=0.0, posinf=35.0, neginf=-35.0)


def fit_l2_logistic(x, y):
    x_design = np.column_stack([np.ones(len(x)), x])
    beta = np.zeros(x_design.shape[1], dtype=float)
    history = []
    for _ in range(LOGIT_MAX_ITER):
        p = sigmoid(linear_predict(x_design, beta))
        error = p - y
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            grad = (x_design.T @ error) / len(y)
        grad = np.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)
        grad[1:] += (L2_STRENGTH / len(y)) * beta[1:]
        new_beta = beta - LOGIT_LEARNING_RATE * grad

        eps = 1e-12
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        loss += (L2_STRENGTH / (2 * len(y))) * np.sum(beta[1:] ** 2)
        history.append(float(loss))

        if np.max(np.abs(new_beta - beta)) < LOGIT_TOL:
            beta = new_beta
            break
        beta = new_beta
    return beta, history


def predict_l2_logistic(x, beta):
    x_design = np.column_stack([np.ones(len(x)), x])
    return sigmoid(linear_predict(x_design, beta))

In [ ]:
logit_beta, logit_history = fit_l2_logistic(x_train_std, y_train)
logit_train_prob = predict_l2_logistic(x_train_std, logit_beta)
logit_test_prob = predict_l2_logistic(x_test_std, logit_beta)

logit_train_metrics = metrics(y_train, logit_train_prob)
logit_test_metrics = metrics(y_test, logit_test_prob)

print("Iterations:", len(logit_history))
print("Final train loss:", round(logit_history[-1], 4))
print("Test metrics:")
print(pd.Series(logit_test_metrics).to_string())

In [ ]:
test_ids = model_df.iloc[test_idx]["C0000100"].astype(int).to_numpy()

logit_pred_df = pd.DataFrame({
    "C0000100": test_ids,
    "target": TARGET,
    "model": "l2_logistic_regression_numpy",
    "y_true": y_test,
    "probability": logit_test_prob,
    "prediction": (logit_test_prob >= 0.5).astype(int),
})
logit_pred_df["correct"] = logit_pred_df["prediction"] == logit_pred_df["y_true"]

logit_coef_df = pd.DataFrame({
    "csv_code": feature_cols,
    "coefficient": logit_beta[1:],
}).merge(
    feature_index[["csv_code", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
)
logit_coef_df["abs_coefficient"] = logit_coef_df["coefficient"].abs()
logit_coef_df = logit_coef_df.sort_values("abs_coefficient", ascending=False)

logit_metrics_df = pd.DataFrame([
    {"split": "train", **logit_train_metrics},
    {"split": "test", **logit_test_metrics},
])

logit_pred_df.to_csv(OUT_DIR / "l2_logistic_regression_predictions.csv", index=False)
logit_coef_df.to_csv(OUT_DIR / "l2_logistic_regression_coefficients.csv", index=False)
logit_metrics_df.to_csv(OUT_DIR / "l2_logistic_regression_metrics.csv", index=False)

logit_summary = {
    "target": TARGET,
    "model": "l2_logistic_regression_numpy",
    "n_total": int(len(model_df)),
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_features": int(len(feature_cols)),
    "test_size": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "l2_strength": L2_STRENGTH,
    "learning_rate": LOGIT_LEARNING_RATE,
    "iterations": len(logit_history),
    "final_train_loss": logit_history[-1],
    "train_metrics": logit_train_metrics,
    "test_metrics": logit_test_metrics,
}
(OUT_DIR / "l2_logistic_regression_summary.json").write_text(json.dumps(logit_summary, indent=2))

logit_coef_df.head(15)

# Model 2: Gradient Boosting with Decision Stumps

This model is a simple nonlinear tabular baseline. Each boosting step fits one decision stump to the residuals.

It is intentionally lightweight. In a final production-style analysis, this could be replaced by `sklearn.GradientBoostingClassifier`, `HistGradientBoostingClassifier`, XGBoost, or LightGBM.

In [ ]:
N_ESTIMATORS = 150
GB_LEARNING_RATE = 0.05
MAX_THRESHOLDS_PER_FEATURE = 24
MIN_LEAF_SIZE = 30


def logit(p):
    p = min(max(float(p), 1e-6), 1 - 1e-6)
    return float(np.log(p / (1 - p)))


def candidate_thresholds(values):
    unique = np.unique(values)
    if len(unique) <= 1:
        return np.array([])
    if len(unique) <= MAX_THRESHOLDS_PER_FEATURE:
        return (unique[:-1] + unique[1:]) / 2
    qs = np.linspace(0.02, 0.98, MAX_THRESHOLDS_PER_FEATURE)
    return np.unique(np.quantile(values, qs))


def fit_best_stump(x, residual, feature_names):
    best = {
        "feature_index": None,
        "feature": None,
        "threshold": None,
        "left_value": 0.0,
        "right_value": 0.0,
        "loss": np.inf,
    }
    n = len(residual)
    for j in range(x.shape[1]):
        values = x[:, j]
        for threshold in candidate_thresholds(values):
            left = values <= threshold
            n_left = int(left.sum())
            n_right = n - n_left
            if n_left < MIN_LEAF_SIZE or n_right < MIN_LEAF_SIZE:
                continue
            left_value = float(residual[left].mean())
            right_value = float(residual[~left].mean())
            pred = np.where(left, left_value, right_value)
            loss = float(np.mean((residual - pred) ** 2))
            if loss < best["loss"]:
                best = {
                    "feature_index": j,
                    "feature": feature_names[j],
                    "threshold": float(threshold),
                    "left_value": left_value,
                    "right_value": right_value,
                    "loss": loss,
                }
    if best["feature_index"] is None:
        raise RuntimeError("Could not find a valid stump split.")
    return best


def predict_stump(x, stump):
    values = x[:, int(stump["feature_index"])]
    return np.where(values <= stump["threshold"], stump["left_value"], stump["right_value"])


def fit_gradient_boosting(x, y, feature_names):
    base_score = logit(y.mean())
    f = np.full(len(y), base_score, dtype=float)
    stumps = []
    history = []
    for iteration in range(N_ESTIMATORS):
        p = sigmoid(f)
        residual = y - p
        stump = fit_best_stump(x, residual, feature_names)
        f += GB_LEARNING_RATE * predict_stump(x, stump)
        stump["iteration"] = iteration + 1
        stumps.append(stump)

        eps = 1e-12
        p = sigmoid(f)
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        history.append(float(loss))
    return base_score, stumps, history


def predict_gradient_boosting(x, base_score, stumps):
    f = np.full(len(x), base_score, dtype=float)
    for stump in stumps:
        f += GB_LEARNING_RATE * predict_stump(x, stump)
    return sigmoid(f)

In [ ]:
gb_base_score, gb_stumps, gb_history = fit_gradient_boosting(x_train_imputed, y_train, feature_cols)
gb_train_prob = predict_gradient_boosting(x_train_imputed, gb_base_score, gb_stumps)
gb_test_prob = predict_gradient_boosting(x_test_imputed, gb_base_score, gb_stumps)

gb_train_metrics = metrics(y_train, gb_train_prob)
gb_test_metrics = metrics(y_test, gb_test_prob)

print("Estimators:", len(gb_stumps))
print("Final train loss:", round(gb_history[-1], 4))
print("Test metrics:")
print(pd.Series(gb_test_metrics).to_string())

In [ ]:
gb_pred_df = pd.DataFrame({
    "C0000100": test_ids,
    "target": TARGET,
    "model": "gradient_boosting_stumps_numpy",
    "y_true": y_test,
    "probability": gb_test_prob,
    "prediction": (gb_test_prob >= 0.5).astype(int),
})
gb_pred_df["correct"] = gb_pred_df["prediction"] == gb_pred_df["y_true"]

gb_stumps_df = pd.DataFrame(gb_stumps).merge(
    feature_index[["csv_code", "survey_year", "feature_group", "question"]],
    left_on="feature",
    right_on="csv_code",
    how="left",
)

gb_importance_df = (
    gb_stumps_df
    .groupby(["feature", "survey_year", "feature_group", "question"], dropna=False)
    .agg(n_splits=("iteration", "count"), mean_split_loss=("loss", "mean"))
    .reset_index()
    .sort_values(["n_splits", "mean_split_loss"], ascending=[False, True])
)

gb_metrics_df = pd.DataFrame([
    {"split": "train", **gb_train_metrics},
    {"split": "test", **gb_test_metrics},
])

gb_pred_df.to_csv(OUT_DIR / "gradient_boosting_stumps_predictions.csv", index=False)
gb_stumps_df.to_csv(OUT_DIR / "gradient_boosting_stumps_splits.csv", index=False)
gb_importance_df.to_csv(OUT_DIR / "gradient_boosting_stumps_feature_importance.csv", index=False)
gb_metrics_df.to_csv(OUT_DIR / "gradient_boosting_stumps_metrics.csv", index=False)

gb_summary = {
    "target": TARGET,
    "model": "gradient_boosting_stumps_numpy",
    "n_total": int(len(model_df)),
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_features": int(len(feature_cols)),
    "test_size": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "n_estimators": N_ESTIMATORS,
    "learning_rate": GB_LEARNING_RATE,
    "max_thresholds_per_feature": MAX_THRESHOLDS_PER_FEATURE,
    "min_leaf_size": MIN_LEAF_SIZE,
    "initial_score": gb_base_score,
    "final_train_loss": gb_history[-1],
    "train_metrics": gb_train_metrics,
    "test_metrics": gb_test_metrics,
}
(OUT_DIR / "gradient_boosting_stumps_summary.json").write_text(json.dumps(gb_summary, indent=2))

gb_importance_df.head(15)

# Model Comparison

Both models are trained and tested on the same split, with the same pre-2000 predictors and the same 2000-2020 target. This makes them directly comparable to each other and later comparable to BoL/LLM outputs.

In [ ]:
comparison = pd.DataFrame([
    {"model": "L2 Logistic Regression", "split": "train", **logit_train_metrics},
    {"model": "L2 Logistic Regression", "split": "test", **logit_test_metrics},
    {"model": "Gradient Boosting Stumps", "split": "train", **gb_train_metrics},
    {"model": "Gradient Boosting Stumps", "split": "test", **gb_test_metrics},
])

comparison.to_csv(OUT_DIR / "tabular_model_comparison_metrics.csv", index=False)
comparison

## Interpretation

The L2 logistic model is the main interpretable tabular baseline. Gradient boosting is a more flexible nonlinear baseline.

For the final project, these results can be compared with Book-of-Life/LLM predictions generated from the same pre-2000 information. If we later install scikit-learn, the next step would be hyperparameter tuning with a train/validation/test split.